In [2]:
import numpy as np
import os
import mne
import scipy.io
from fooof import FOOOF
import gc
import psutil

from zapline_iter import zapline_until_gone
from superlet import superlet, scale_from_period
from burst_detection import extract_bursts

data_dir = '/home/common/bonaiuto/stop_go_bursts/data/'
output_dir = '/home/qmoreau/schmidt_data/output/'
data_path = os.path.join(data_dir, 'Dataset_1/Study5_EEG_data/')
plot_dir = "/home/qmoreau/schmidt_data/fooof_plots/"
os.makedirs(plot_dir, exist_ok=True)

subject_ids = ['S1', 'S2', 'S3', 'S5', 'S6', 'S7', 'S8', 'S10', 'S11', 'S13', 'S14', 'S15']
epoch_types = ['GO', 'SS', 'FS', 'GO_bl', 'SS_bl', 'FS_bl']
electrodes = ['C3', 'F4']

chunk_size = 25


def check_memory():
    mem_gb = psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024 / 1024
    if mem_gb > 6:
        print(f"  High memory usage: {mem_gb:.1f} GB")
        gc.collect()
    return mem_gb


def process_subject_epoch(subject_id, epoch_type):
    print(f"  Processing {subject_id} {epoch_type}")

    try:
        subject_out_dir = os.path.join(output_dir, subject_id)
        os.makedirs(subject_out_dir, exist_ok=True)

        subject_file = os.path.join(data_path, f"{subject_id}_stop_eeg_emg_ica_brain.set")
        raw = mne.io.read_raw_eeglab(subject_file, eog='auto', preload=False)
        raw.rename_channels({'FP1': 'Fp1', 'FP2': 'Fp2'})
        raw.pick_types(eeg=True)
        raw.load_data()

        ssd_path = os.path.join(data_dir, f'Dataset_1/Study5_BEH_data/{subject_id}_SSD_stop.mat')
        SSD = scipy.io.loadmat(ssd_path)
        SSD_val_ms = np.mean(SSD['num_align2'][:, 2])
        ssd_shift_samples = int(SSD_val_ms / 1000 * 512)
        print(f"  SSD_val = {SSD_val_ms:.1f} ms => {ssd_shift_samples} samples")

        montage = mne.channels.make_standard_montage("easycap-M1")
        raw.set_montage(montage, verbose=False)
        events, event_id = mne.events_from_annotations(raw)

        events_to_add = []
        new_event_id = event_id.copy()
        max_id = max(event_id.values()) if event_id else 0
        new_id_counter = max_id + 1

        for name, original_id in event_id.items():
            if original_id == 1 or original_id == 9:
                shifted_name = name + '_shifted'
                new_event_id[shifted_name] = new_id_counter
                matching_events = events[events[:, 2] == original_id]
                for event in matching_events:
                    shifted_event = event.copy()
                    shifted_event[0] = shifted_event[0] + ssd_shift_samples
                    shifted_event[2] = new_id_counter
                    events_to_add.append(shifted_event)
                new_id_counter += 1

        if events_to_add:
            events_combined = np.vstack([events, np.array(events_to_add)])
            events_combined = events_combined[events_combined[:, 0].argsort()]
        else:
            events_combined = events

        epochs = mne.Epochs(raw, events=events_combined, picks='eeg', event_id=new_event_id,
                            tmin=-1, tmax=2, baseline=None, event_repeated='merge', preload=True)
        del raw
        gc.collect()

        epochs = mne.preprocessing.compute_current_source_density(epochs)

        print(f"  Applying Zapline to all {len(epochs)} epochs...")
        epochs._data, _ = zapline_until_gone(
            epochs.get_data(), target_freq=24, sfreq=epochs.info['sfreq'],
            win_sz=5, spot_sz=1.5, viz=False, prefix=f"{subject_id}_epoch", max_iter=3
        )
        gc.collect()

        try:
            SS          = epochs['L_stopsigSS', 'R_stopsigSS']
            FS          = epochs['L_stopsigFS', 'R_stopsigFS']
            GO          = epochs['L_cueCG_shifted', 'R_cueCG_shifted']
            SS_baseline = epochs['L_cueSS', 'R_cueSS']
            FS_baseline = epochs['L_cueFS', 'R_cueFS']
            GO_baseline = epochs['L_cueCG', 'R_cueCG']
        except KeyError as e:
            print(f"    Missing events for {subject_id}: {e}")
            return None

        conditions = {
            'SS': SS, 'FS': FS, 'GO': GO,
            'SS_bl': SS_baseline, 'FS_bl': FS_baseline, 'GO_bl': GO_baseline
        }

        if epoch_type not in conditions:
            print(f"    Skipping invalid epoch type: {epoch_type}")
            return None

        condition = conditions[epoch_type]
        del epochs, SS, FS, GO, SS_baseline, FS_baseline, GO_baseline
        gc.collect()

        results = []

        for electrode in electrodes:
            print(f"    Processing electrode: {electrode}")
            try:
                ch_idx = condition.ch_names.index(electrode)
            except ValueError:
                print(f"      Electrode {electrode} not found, skipping")
                continue

            trials = condition.get_data()[:, ch_idx, :]
            n_trials = len(trials)
            if n_trials == 0:
                print(f"      No trials for {electrode}")
                continue

            foi = np.linspace(1, 120, 120)
            scales = scale_from_period(1 / foi)
            search_range = np.where((foi >= 10) & (foi <= 33))[0]
            beta_lims = [13, 30]

            tf_sum = None
            tf_count = 0
            all_bursts = []
            all_beta_pow = []
            all_tf_chunks = []

            for start_idx in range(0, n_trials, chunk_size):
                end_idx = min(start_idx + chunk_size, n_trials)
                chunk_trials = trials[start_idx:end_idx]
                print(f"      TFR trials {start_idx} to {end_idx} of {n_trials}")

                tf_chunk = np.array([
                    np.abs(superlet(trial, condition.info['sfreq'], scales, 40, 4, adaptive=True))
                    for trial in chunk_trials
                ])

                beta_pow_chunk = np.mean(tf_chunk[:, (foi >= 13) & (foi <= 30), :], axis=1)
                all_beta_pow.append(beta_pow_chunk)
                all_tf_chunks.append(tf_chunk.astype(np.float32))

                if tf_sum is None:
                    tf_sum = np.sum(tf_chunk, axis=(0, 2))
                else:
                    tf_sum += np.sum(tf_chunk, axis=(0, 2))
                tf_count += tf_chunk.shape[0] * tf_chunk.shape[2]

                del chunk_trials, tf_chunk
                gc.collect()
                check_memory()

            # Save beta power for all epoch types (used for baseline correction downstream)
            if all_beta_pow:
                beta_pow_all = np.concatenate(all_beta_pow, axis=0)
                np.savez(
                    os.path.join(subject_out_dir, f'{subject_id}_{epoch_type}_{electrode}_beta_power.npz'),
                    subject_id=subject_id, epoch_type=epoch_type,
                    electrode=electrode, beta_pow=beta_pow_all, time=condition.times
                )

            if all_tf_chunks:
                tf_all = np.concatenate(all_tf_chunks, axis=0)
                np.savez(
                    os.path.join(subject_out_dir, f'{subject_id}_{epoch_type}_{electrode}_tf.npz'),
                    subject_id=subject_id, epoch_type=epoch_type,
                    electrode=electrode, tf=tf_all, freqs=foi, time=condition.times
                )

            # Burst extraction runs for ALL epoch types including baselines.
            # Baseline bursts are tagged with '_bl' in bursts['condition'] so the
            # downstream PCA script can exclude them from the PCA fit with:
            #   stop_mask = ~np.char.endswith(bursts['condition'].astype(str), '_bl')
            # but they are retained in the saved results for use as burst rate baselines.
            average_psd = tf_sum / tf_count
            try:
                ff = FOOOF()
                ff.fit(foi, average_psd, [3, 50])
                interpolated_ap = 10 ** (
                    ff.aperiodic_params_[0] - ff.aperiodic_params_[1] * np.log10(foi)
                )
                print(f"      FOOOF fit OK  aperiodic params: {ff.aperiodic_params_}")
            except Exception as e:
                print(f"      FOOOF fitting failed ({e}), falling back to flat baseline")
                interpolated_ap = np.ones(len(foi)) * np.mean(average_psd)

            for chunk_idx, (start_idx, tf_chunk) in enumerate(
                zip(range(0, n_trials, chunk_size), all_tf_chunks)
            ):
                end_idx = min(start_idx + chunk_size, n_trials)
                chunk_trials = trials[start_idx:end_idx]

                try:
                    bursts = extract_bursts(chunk_trials, 
                                            tf_chunk[:, search_range], 
                                            condition.times,
                                            foi[search_range], 
                                            beta_lims,
                                            interpolated_ap[search_range].reshape(-1, 1), 
                                            512)
                    
                    n_bursts = len(bursts['trial']) if 'trial' in bursts else 0
                    if n_bursts > 0:
                        bursts['condition']  = np.tile(epoch_type, n_bursts)
                        bursts['electrode']  = np.tile(electrode, n_bursts)
                        bursts['subject']    = np.tile(subject_id, n_bursts)
                        bursts['experiment'] = np.tile('Exp1', n_bursts)
                        bursts['epochs']     = np.tile('Stop', n_bursts)
                        bursts['trial']     += start_idx
                        all_bursts.append(bursts)
                except Exception as e:
                    print(f"        Burst extraction failed for chunk {chunk_idx}: {e}")

            if all_bursts:
                # waveform_times is a shared 1D time axis, not a per-burst array.
                # Take it from the first chunk only - concatenating it across chunks
                # causes it to be repeated N-chunks times.
                shared_keys = {'waveform_times'}
                combined_bursts = {}
                for key in all_bursts[0].keys():
                    if key in shared_keys:
                        combined_bursts[key] = all_bursts[0][key]
                    else:
                        combined_bursts[key] = np.concatenate(
                            [b[key] for b in all_bursts if key in b]
                        )
                results.append(combined_bursts)
                print(f"      Found {len(combined_bursts.get('trial', []))} bursts")

            del trials, tf_sum, all_bursts, all_beta_pow, all_tf_chunks
            gc.collect()

        print(f"  Completed {subject_id} {epoch_type}")
        return results

    except Exception as e:
        print(f"  ERROR processing {subject_id} {epoch_type}: {str(e)}")
        import traceback
        traceback.print_exc()
        return None


# Main loop
output_file = f'{output_dir}/bursts_all_results_laplac.npy'
bursts_all_results_laplac = []
processed_combos = set()
total_combinations = len(subject_ids) * len(epoch_types)
processed_count = 0

print(f"Processing {total_combinations} combinations...")

for subject_id in subject_ids:
    print(f"\nProcessing subject {subject_id}...")
    for epoch_type in epoch_types:
        processed_count += 1
        print(f"Progress: {processed_count}/{total_combinations} ({100*processed_count/total_combinations:.1f}%)")

        combo_key = (subject_id, epoch_type)
        if combo_key in processed_combos:
            print(f"  Skipping duplicate: {subject_id} {epoch_type}")
            continue
        processed_combos.add(combo_key)

        result = process_subject_epoch(subject_id, epoch_type)

        if result:
            bursts_all_results_laplac.extend(result)
            print(f"  Added {len(result)} results")
        else:
            print(f"  No results returned")

        gc.collect()

        if processed_count % 10 == 0:
            np.save(f'{output_dir}/bursts_intermediate_{processed_count}.npy', bursts_all_results_laplac)
            print(f"  Saved intermediate checkpoint")

        check_memory()

np.save(output_file, bursts_all_results_laplac)
print(f"\nFinal results saved: {output_file}")
print(f"Total results: {len(bursts_all_results_laplac)}")
print(f"Final memory usage: {check_memory():.1f} GB")

Processing 72 combinations...

Processing subject S1...
Progress: 1/72 (1.4%)
  Processing S1 GO
Reading /home/common/bonaiuto/stop_go_bursts/data/Dataset_1/Study5_EEG_data/S1_stop_eeg_emg_ica_brain.fdt
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 2016056  =      0.000 ...  3937.609 secs...


/tmp/ipykernel_64573/2514886536.py:42: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(subject_file, eog='auto', preload=False)
/tmp/ipykernel_64573/2514886536.py:42: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(subject_file, eog='auto', preload=False)
/tmp/ipykernel_64573/2514886536.py:42: RuntimeWarning: Not setting positions of 3 eog channels found in montage:
['HEOG L', 'HEOG R', 'VEOG']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(subject_file, eog='auto', preload=False)


  SSD_val = 132.0 ms => 67 samples
Used Annotations descriptions: ['L_cueCG', 'L_cueFS', 'L_cueGE', 'L_cueGM', 'L_cueSS', 'L_resp', 'L_stopsigFS', 'L_stopsigSS', 'R_cueCG', 'R_cueFS', 'R_cueGE', 'R_cueGM', 'R_cueSS', 'R_cue_stop', 'R_resp', 'R_stopsig', 'R_stopsigFS', 'R_stopsigSS', 'boundary', 'fix']
Multiple event values for single event times found. Creating new event value to reflect simultaneous events.
Not setting metadata
5890 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 5890 events and 1537 original time points ...
1 bad epochs dropped
Fitted sphere radius:         95.0 mm
Origin head coordinates:      0.0 0.0 40.1 mm
Origin device coordinates:    0.0 0.0 40.1 mm
  Applying Zapline to all 5889 epochs...
Iteration: 0 Power above the fit: 0.07800492607541909
Power of components removed by DSS: 0.05
Iteration: 1 Power above the fit: 0.043268081525704605
Power of components removed by DSS: 0.04
Iteration: 2 Powe

/home/qmoreau/schmidt_data/stop_go_bursts/burst_detection.py:79: RuntimeWarning: All-NaN axis encountered
  horiz = np.nanmin([left_loc, right_loc])


      Found 4771 bursts
    Processing electrode: F4
      TFR trials 0 to 25 of 1063
      TFR trials 25 to 50 of 1063
      TFR trials 50 to 75 of 1063
      TFR trials 75 to 100 of 1063
      TFR trials 100 to 125 of 1063
      TFR trials 125 to 150 of 1063
      TFR trials 150 to 175 of 1063
      TFR trials 175 to 200 of 1063
      TFR trials 200 to 225 of 1063
      TFR trials 225 to 250 of 1063
      TFR trials 250 to 275 of 1063
      TFR trials 275 to 300 of 1063
      TFR trials 300 to 325 of 1063
      TFR trials 325 to 350 of 1063
      TFR trials 350 to 375 of 1063
      TFR trials 375 to 400 of 1063
      TFR trials 400 to 425 of 1063
      TFR trials 425 to 450 of 1063
      TFR trials 450 to 475 of 1063
      TFR trials 475 to 500 of 1063
      TFR trials 500 to 525 of 1063
      TFR trials 525 to 550 of 1063
      TFR trials 550 to 575 of 1063
      TFR trials 575 to 600 of 1063
      TFR trials 600 to 625 of 1063
      TFR trials 625 to 650 of 1063
      TFR trials 65

...

  SSD_val = 165.8 ms => 84 samples
Used Annotations descriptions: ['L_cueCG', 'L_cueFS', 'L_cueGE', 'L_cueGM', 'L_cueSS', 'L_resp', 'L_stopsigFS', 'L_stopsigSS', 'R_cueCG', 'R_cueFS', 'R_cueGE', 'R_cueGM', 'R_cueSS', 'R_resp', 'R_stopsigFS', 'R_stopsigSS', 'boundary', 'fix']
Multiple event values for single event times found. Creating new event value to reflect simultaneous events.
Not setting metadata
6211 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 6211 events and 1537 original time points ...
6 bad epochs dropped
Fitted sphere radius:         95.0 mm
Origin head coordinates:      0.0 0.0 40.1 mm
Origin device coordinates:    0.0 0.0 40.1 mm
  Applying Zapline to all 6205 epochs...
Iteration: 0 Power above the fit: 0.02465361765160823
Power of components removed by DSS: 0.01
Iteration: 1 Power above the fit: 0.006482915911867004
Power of components removed by DSS: 0.01
Iteration: 2 Power above the fit: -0.0035263

/home/qmoreau/schmidt_data/stop_go_bursts/burst_detection.py:79: RuntimeWarning: All-NaN axis encountered
  horiz = np.nanmin([left_loc, right_loc])


      Found 631 bursts
  Completed S15 SS_bl
  Added 2 results
Progress: 72/72 (100.0%)
  Processing S15 FS_bl
Reading /home/common/bonaiuto/stop_go_bursts/data/Dataset_1/Study5_EEG_data/S15_stop_eeg_emg_ica_brain.fdt
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Reading 0 ... 1976434  =      0.000 ...  3860.223 secs...


/tmp/ipykernel_64573/2514886536.py:42: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(subject_file, eog='auto', preload=False)
/tmp/ipykernel_64573/2514886536.py:42: RuntimeWarning: Not setting positions of 3 eog channels found in montage:
['HEOG L', 'HEOG R', 'VEOG']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(subject_file, eog='auto', preload=False)


  SSD_val = 165.8 ms => 84 samples
Used Annotations descriptions: ['L_cueCG', 'L_cueFS', 'L_cueGE', 'L_cueGM', 'L_cueSS', 'L_resp', 'L_stopsigFS', 'L_stopsigSS', 'R_cueCG', 'R_cueFS', 'R_cueGE', 'R_cueGM', 'R_cueSS', 'R_resp', 'R_stopsigFS', 'R_stopsigSS', 'boundary', 'fix']
Multiple event values for single event times found. Creating new event value to reflect simultaneous events.
Not setting metadata
6211 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 6211 events and 1537 original time points ...
6 bad epochs dropped
Fitted sphere radius:         95.0 mm
Origin head coordinates:      0.0 0.0 40.1 mm
Origin device coordinates:    0.0 0.0 40.1 mm
  Applying Zapline to all 6205 epochs...
Iteration: 0 Power above the fit: 0.02465361765160823
Power of components removed by DSS: 0.01
Iteration: 1 Power above the fit: 0.006482915911867004
Power of components removed by DSS: 0.01
Iteration: 2 Power above the fit: -0.0035263

In [9]:
import scipy.io
import numpy as np

subject_id = 'S1'
ssd_path = f'/home/common/bonaiuto/stop_go_bursts/data/Dataset_1/Study5_BEH_data/{subject_id}_SSD_stop.mat'

SSD = scipy.io.loadmat(ssd_path)
print(SSD.keys())
print(SSD['num_align2'][:, 2])
print(f"Mean: {np.mean(SSD['num_align2'][:, 2]):.4f}")

dict_keys(['__header__', '__version__', '__globals__', 'num_align2'])
[200 150 200 150 200 250 200 100 150 100 250 150 200 250 200 200 150 200
 150 150 200 200 150 150 200 200 150 200 150 200 250 150 200 150 200 150
 100 100 150 150 100 150 200 150 200 100 150 100 250 150 200 150 100 150
 100 150 200 200 150 150 200 200 250 200 150 100 150 150 100 200 150  50
 200 150 100 150 200 200 150 150 200 150 200 200 150 250 200 200 150 100
 150 200 250 200 150 100 250 200 150 150 100 100 150 200 250 150 200 150
 200 150 200 100  50 150 100 150 100 100 150 150 200 150 100 150 100 200
 150 150 100 150 200 200 150 100 150 150 100 200 150 150 100 150 200 200
 150 250 200 200 150 200 150 250 200 150 200 200 150 150 100 150 100 200
 150  50 100 100 150 150 100 150 100 200 150 150 200 100 150 150 200 150
 100 200 150 200 150 150 100 100 150 100 150 150 200 150 100 150 100 100
  50 150 100 100 150 100  50 100  50 150 200 100 150 100 150 200 250  50
 100 200  50 150 100 200  50 150 100 100 150 200 150  